<img src="images/logos/esgf2-us.png" width=250 alt="ESGF logo"></img>

# Complex Searching with `intake` and analysing employing `xarray` 

## Overview

This tutorial we will present access multiple historical (as an example here) data available and analyze  using `intake`. Put them in a dictionary format employing `xarray` and plotting simple area average time series over a specific region. 

## Imports

In [ ]:
import warnings
import intake
from distributed import Client
from matplotlib import pyplot as plt
import xarray as xr
import dask
xr.set_options(display_style='html')
warnings.filterwarnings("ignore")

In [ ]:
cat_url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(cat_url)
col

In [ ]:
cat = col.search(experiment_id=["historical"],
    variable_id = ["tas"],
    member_id = ["r1i1p1f1"],
    table_id = ["Amon",], 
    source_id = [ "CMCC-ESM2", "CanESM5", "CESM2", "CESM2-FV2", ]
)

In [ ]:
cat.df

In [ ]:
dset_dict = cat.to_dataset_dict(zarr_kwargs={'consolidated': True})
list(dset_dict.keys())

In [ ]:
ds = {}

for key in dset_dict.keys():
    # Sort the dataset by time
    sorted_dataset = dset_dict[key].sortby("time")
    
    # Subset data for years 1900-2000
    ds[key] = sorted_dataset.sel(time=slice("1900", "2000"))
        
    # Optional: Print a message indicating dataset processing
    print(f"Processing dataset: {key}")


**`ds` now contains subset of datasets for each key in dset_dict** 

**Let's check ds**

In [ ]:
ds

### Calculate regional mean for each dataset and visualizing time series

In [ ]:
regn_mean = {} 
for key in dset_dict.keys():
    regn_mean[key] = ds[key]['tas'].sel(lon=slice(65, 100), lat=slice(5, 25)).mean(dim=['lon', 'lat']).squeeze()


In [ ]:
regn_mean

In [ ]:
plt.rcParams['figure.figsize'] = [15, 4]
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['font.size'] = 10
plt.rcParams['font.weight'] = 'bold'


#### Visualizing the regional mean for each dataset

In [ ]:
for key, regm in regn_mean.items():
    source_id = key.split('.')[2]
    regm.plot(label=source_id)
    plt.title(f"Mean Surface Air Temperature for {source_id}")
    plt.xlabel('Time')
    plt.ylabel('Temperature (K)')
    plt.legend()
    plt.show();

### Calculating annual mean for each dataset  and visualizing time series

In [ ]:
annual_mean = {}
for key, regm in regn_mean.items():
    annual_mean[key] = regm.resample(time='Y').mean()

In [ ]:
annual_mean

#### Visualizing the regional annual mean for each dataset

In [ ]:
for key, anmn in annual_mean.items():
    source_id = key.split('.')[2]
    anmn.plot(label=source_id)
    plt.title(f"Mean Annual Surface Air Temperature for {source_id}")
    plt.xlabel('Time')
    plt.ylabel('Temperature (K)')
    plt.legend()
    plt.show();

#### Visualizing the regional annual mean for each dataset in a single panel


In [ ]:
# Create the plot
plt.figure(figsize=(12, 6))

# Plotting the annual mean for each dataset on the same plot
for key, annm in annual_mean.items():
    source_id = key.split('.')[2]
    annm.plot(label=source_id)

plt.title("Annual Mean Surface Air Temperature (Regional)")
plt.xlabel('Time')
plt.ylabel('Temperature (K)')
plt.legend()
plt.show();

### References 
[Global Mean Surface Temperature](https://projectpythia.org/cmip6-cookbook/notebooks/example-workflows/gmst/)